# 39 CPU audio audit + baseline compare

音声研究用の再確認 notebook です。clip に本当に音声 stream と contact-window 音声があるかを `ffprobe`/`ffmpeg` で監査し、音あり clip だけで raw/enhanced audio baseline を作り直します。

Creates: `features/{audio_audit_feature_id}/audio_presence_manifest.parquet`, `features/{audio_audit_feature_id}/audio_valid_clips_v1.parquet`, `reports/audio_audit/{audio_presence_audit_id}/`, `predictions/{audio_*_valid_run_id}/`, `reports/audio_baseline_compare/{audio_baseline_compare_report_id}/`.

Required inputs: `manifests/bbe_events_v1.parquet`, `clips/{full_run_id}/clips_v1.parquet`, existing baseline `predictions/*/predictions_v1.parquet`.

Runtime: CPU. If `recover_missing_from_sources=True`, ffmpeg tries to recover short wav windows from raw/source video paths already present in the clip manifest.

In [ ]:
from pathlib import Path
import json
import os
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or already mounted:', exc)

os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff_with_audio')

repo_dir = Path(os.environ['BATTING_CODE_ROOT'])
if not (repo_dir / 'src' / 'sport_pipeline').exists():
    repo_dir = Path.cwd()
sys.path.insert(0, str(repo_dir / 'src'))

profile_path = repo_dir / 'configs' / 'runs' / os.environ['BASEBALL_VISION_RUN_PROFILE']
profile = json.loads(profile_path.read_text(encoding='utf-8'))
BASE_DIR = Path(profile['paths']['base_dir'])
CACHE_DIR = Path(profile['paths'].get('cache_dir', '/content/cache/baseball_vision'))
RUN_IDS = profile['run_ids']
NS = profile['artifact_namespace']
AUDIT_EXEC = profile['execution'].get('audio_audit', {})
AUDIO_EXEC = profile['execution']['audio_impact']
COMPARE_EXEC = profile['execution'].get('audio_baseline_compare', {})

print('repo_dir=', repo_dir)
print('BASE_DIR=', BASE_DIR)
print('full_run_id=', RUN_IDS['full_run_id'])

In [ ]:
required = [
    BASE_DIR / 'manifests' / 'bbe_events_v1.parquet',
    BASE_DIR / 'clips' / RUN_IDS['full_run_id'] / 'clips_v1.parquet',
]
for path in required:
    print(('OK      ' if path.exists() else 'MISSING '), path)
if not all(path.exists() for path in required):
    raise FileNotFoundError('Required artifact missing. Run Statcast/download/CV preprocessing notebooks first.')

In [ ]:
from sport_pipeline.audio.audit import run_audio_presence_audit

audit_outputs = run_audio_presence_audit(
    BASE_DIR,
    clip_run_id=RUN_IDS['full_run_id'],
    audit_id=RUN_IDS.get('audio_presence_audit_id', 'audio_presence_mlb_2024_2026_v2'),
    max_clips=AUDIT_EXEC.get('max_clips'),
    recover_missing_from_sources=AUDIT_EXEC.get('recover_missing_from_sources', False),
    preview_examples=AUDIT_EXEC.get('preview_examples', 8),
    output_suffix='.parquet',
)
print(json.dumps({k: str(v) for k, v in audit_outputs.items()}, indent=2, ensure_ascii=False))
valid_clips_path = audit_outputs['audio_valid_clips']
if not valid_clips_path.exists():
    raise FileNotFoundError(f'audio_valid_clips was not created: {valid_clips_path}')

In [ ]:
from sport_pipeline.audio.impact import run_audio_impact_baseline

COMMON = dict(
    base_dir=BASE_DIR,
    clip_run_id=RUN_IDS['full_run_id'],
    clips_path=valid_clips_path,
    max_clips=AUDIO_EXEC.get('max_clips'),
    require_non_empty=AUDIO_EXEC.get('require_non_empty', True),
    output_suffix='.parquet',
    cache_dir=CACHE_DIR,
    cache_inputs=AUDIO_EXEC.get('cache_inputs', True),
    cache_num_workers=AUDIO_EXEC.get('cache_num_workers', 4),
    cache_min_free_disk_gb=AUDIO_EXEC.get('cache_min_free_disk_gb', 30),
    cache_max_file_mb=AUDIO_EXEC.get('cache_max_file_mb', 256),
    valid_audio_only=True,
)

audio_outputs = {}
audio_outputs['raw_valid'] = run_audio_impact_baseline(
    prediction_run_id=RUN_IDS.get('audio_raw_valid_run_id', 'audio_raw_valid_impact_mlb_2024_2026_v2'),
    audio_feature_id=NS.get('audio_raw_valid_feature_id', 'audio_raw_valid_impact_mlb_2024_2026_v2'),
    preprocessing_mode='raw',
    **COMMON,
)
audio_outputs['enhanced_valid'] = run_audio_impact_baseline(
    prediction_run_id=RUN_IDS.get('audio_enhanced_valid_run_id', 'audio_enhanced_valid_impact_mlb_2024_2026_v2'),
    audio_feature_id=NS.get('audio_enhanced_valid_feature_id', 'audio_enhanced_valid_impact_mlb_2024_2026_v2'),
    preprocessing_mode='enhanced',
    **COMMON,
)
print(json.dumps({k: {kk: str(vv) for kk, vv in v.items()} for k, v in audio_outputs.items()}, indent=2, ensure_ascii=False))

In [ ]:
from sport_pipeline.reports.audio_baseline_compare import write_audio_baseline_comparison_report

audio_run_ids = COMPARE_EXEC.get('audio_run_ids') or [
    RUN_IDS.get('audio_raw_valid_run_id', 'audio_raw_valid_impact_mlb_2024_2026_v2'),
    RUN_IDS.get('audio_enhanced_valid_run_id', 'audio_enhanced_valid_impact_mlb_2024_2026_v2'),
]
baseline_run_ids = COMPARE_EXEC.get('baseline_run_ids') or [
    RUN_IDS['context_run_id'],
    RUN_IDS['video_lightweight_run_id'],
    RUN_IDS['fusion_run_id'],
]
compare_outputs = write_audio_baseline_comparison_report(
    BASE_DIR,
    report_id=RUN_IDS.get('audio_baseline_compare_report_id', 'audio_baseline_compare_mlb_2024_2026_v2'),
    audio_run_ids=audio_run_ids,
    baseline_run_ids=baseline_run_ids,
    min_intersection=COMPARE_EXEC.get('min_intersection', 3),
    output_suffix='.parquet',
)
print(json.dumps({k: str(v) for k, v in compare_outputs.items()}, indent=2, ensure_ascii=False))

In [ ]:
print('\nPrimary outputs to inspect:')
print('audio audit html:', audit_outputs['audio_audit_html'])
print('audio valid clips:', audit_outputs['audio_valid_clips'])
print('raw valid report:', audio_outputs['raw_valid'].get('audio_report_html'))
print('enhanced valid report:', audio_outputs['enhanced_valid'].get('audio_report_html'))
print('baseline compare html:', compare_outputs['audio_baseline_compare_html'])
print('\nIf valid_audio_clips is too small, set recover_missing_from_sources=true in the run profile only after confirming raw/source video paths exist and have audio streams.')